# Imbalanced Classification Evaluation
Run cells from top to bottom. This notebook generates an imbalanced dataset, trains a baseline classifier, evaluates thresholds, and saves plots.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project folder:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, ConfusionMatrixDisplay
from src.custom_metrics import calculate_metrics, plot_roc_curve, plot_precision_recall_curve

## 1. Create an imbalanced dataset

In [ ]:
X, y = make_classification(n_samples=3000, n_features=12, n_informative=7,
    n_redundant=2, weights=[0.90, 0.10], random_state=42)
print(pd.Series(y).value_counts().sort_index().rename("count"))

## 2. Train baseline classifier without imbalance handling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight=None, random_state=42))
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)
print(classification_report(y_test, y_pred, zero_division=0))

## 3. Custom metrics, ROC and Precision-Recall curves

In [ ]:
print(calculate_metrics(y_test, y_prob, threshold=0.5))
plot_roc_curve(y_test, y_prob)
plt.show()
plot_precision_recall_curve(y_test, y_prob)
plt.show()

## 4. Compare classification thresholds

In [ ]:
threshold_table = pd.DataFrame([
    calculate_metrics(y_test, y_prob, threshold=round(float(t), 2))
    for t in np.arange(0.1, 0.91, 0.1)
])
display(threshold_table[["threshold", "precision", "recall", "f1", "fp", "fn"]])

## 5. Compare micro, macro and weighted F1

In [ ]:
print("Micro F1:", f1_score(y_test, y_pred, average="micro", zero_division=0))
print("Macro F1:", f1_score(y_test, y_pred, average="macro", zero_division=0))
print("Weighted F1:", f1_score(y_test, y_pred, average="weighted", zero_division=0))

## 6. Confusion matrix and save outputs

In [ ]:
results = ROOT / "results"
results.mkdir(exist_ok=True)
fig, ax = plt.subplots()
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["Majority 0", "Minority 1"], values_format="d", ax=ax)
ax.set_title("Confusion Matrix (threshold = 0.50)")
fig.tight_layout()
fig.savefig(results / "confusion_matrix.png", dpi=150)
plt.show()
threshold_table.to_csv(results / "threshold_metrics.csv", index=False)
plot_roc_curve(y_test, y_prob, results / "roc_curve.png")
plt.close("all")
plot_precision_recall_curve(y_test, y_prob, results / "precision_recall_curve.png")
plt.close("all")
print("Saved outputs to", results.resolve())